#Debug de pipes para el entrenamiento

In [1]:
%cd /home/sven-klein-plarre/Documentos/Universidad/Magister/Tesis/Repositorio-GERUMO/gerumo

/home/sven-klein-plarre/Documentos/Universidad/Magister/Tesis/Repositorio-GERUMO/gerumo


In [2]:
from gerumo import *
import json
from ctapipe.instrument import CameraGeometry

/home/sven-klein-plarre/Documentos/Universidad/Magister/Tesis/Repositorio-GERUMO/gerumo/gerumo/data/pixels_positions/ML1/raw/MST_NectarCam.npy not found.
File ML1/raw/MST_NectarCam not found
/home/sven-klein-plarre/Documentos/Universidad/Magister/Tesis/Repositorio-GERUMO/gerumo/gerumo/data/pixels_positions/ML1/simple/MST_NectarCam.npy not found.
File ML1/simple/MST_NectarCam not found
/home/sven-klein-plarre/Documentos/Universidad/Magister/Tesis/Repositorio-GERUMO/gerumo/gerumo/data/pixels_positions/ML1/simple_shift/MST_NectarCam.npy not found.
File ML1/simple_shift/MST_NectarCam not found
/home/sven-klein-plarre/Documentos/Universidad/Magister/Tesis/Repositorio-GERUMO/gerumo/gerumo/data/pixels_positions/ML1/time/MST_NectarCam.npy not found.
File ML1/time/MST_NectarCam not found
/home/sven-klein-plarre/Documentos/Universidad/Magister/Tesis/Repositorio-GERUMO/gerumo/gerumo/data/pixels_positions/ML1/time_shift/MST_NectarCam.npy not found.
File ML1/time_shift/MST_NectarCam not found
/home

In [4]:
config_path = "/home/sven-klein-plarre/Documentos/Universidad/Magister/Tesis/Repositorio-GERUMO/gerumo/train/config/local/alt_az/umonna_LST.json"
with open(config_path) as cfg_file:
    config = json.load(cfg_file)

In [22]:
telescope = config["telescope"]
version = config["version"]
preprocessing_parameters = config.get("preprocessing", {})
camera_parameters = preprocessing_parameters["CameraPipe"]
train_events_csv    = config["train_events_csv"]
train_telescope_csv = config["train_telescope_csv"]
replace_folder_train = config["replace_folder_train"]
min_observations = config["min_observations"]
targets = config["targets"]
target_mode = config["target_mode"]
input_image_mode = config["input_image_mode"]
input_image_mask = config["input_image_mask"]
input_features = config["input_features"]

target_mode_config = get_target_mode_config(config, target_mode)
target_domains_list = target_mode_config["target_domains"]

target_domains = {target: target_domain for target, target_domain in zip(targets, target_domains_list)}

In [6]:
train_dataset = load_dataset(train_events_csv, train_telescope_csv, replace_folder_train)
train_dataset = aggregate_dataset(train_dataset, az=True, log10_mc_energy=True)
train_dataset = filter_dataset(train_dataset, version, telescope, min_observations, target_domains)

In [7]:
batch_dataset = train_dataset.iloc[0:19]

In [16]:
camera_pipe = CameraPipe(telescope_type=telescope, version=version, **camera_parameters)
cameras = load_cameras(batch_dataset, version=version)
telescope_types = ["LST"]*len(batch_dataset)

In [10]:
MST_flash = load_camera_geometry("MST_NectarCam","DL1")
LST_flash = load_camera_geometry("LST_LSTCam","DL1")
geometry = CameraGeometry.from_name("NectarCam")

In [11]:
cam = camera_pipe(cameras)

In [12]:
a = cam[0][0]

In [13]:
pix_L = PIXELS_POSITION[version]["simple_shift"]["LST_LSTCam"]
print("X_left: ",min(pix_L[0]), ", ", max(pix_L[0]))
print("X_right: ", min(pix_L[1]), ", ", max(pix_L[1]))
print("Y: ", min(pix_L[2]), ", ", max(pix_L[2]))

X_left:  0 ,  46
X_right:  0 ,  46
Y:  0 ,  54


In [14]:
#x_left, x_right, y
pix = PIXELS_POSITION[version]["simple_shift"][telescope]
print("X_left: ",min(pix[0]), ", ", max(pix[0]))
print("X_right: ", min(pix[1]), ", ", max(pix[1]))
print("Y: ", min(pix[2]), ", ", max(pix[2]))

X_left:  0 ,  46
X_right:  0 ,  46
Y:  0 ,  54


In [17]:
img = cameras_to_images(cameras, telescope_types, input_image_mode, input_image_mask, version=version)

In [25]:
test_b_dataset = batch_dataset[input_features]
#t_b_d_2 = scaler.transfrm(test_b_dataset)
print(type(test_b_dataset))

<class 'pandas.core.frame.DataFrame'>


In [26]:
b = [img, test_b_dataset]

In [29]:
print(type(b[0]),type(b[1]))
print(b[1]["y"])

<class 'list'> <class 'pandas.core.frame.DataFrame'>
14   -64.540001
15    50.490002
18    50.490002
19    66.139999
24   -52.070000
25    66.139999
33   -64.540001
34    50.490002
35    66.139999
36   -52.070000
37   -64.540001
38    50.490002
39    66.139999
40   -52.070000
48   -52.070000
49    66.139999
50    50.490002
51   -64.540001
63   -64.540001
Name: y, dtype: float64
